# v31 — マルチ参照 cross-attention アライナー(PF完全不使用, feasibility probe)
**判定基準(ユーザー設定): overall RMSE < 10 が出なければ採用の余地なし**(PFフリー床 ≈13)。

- 参照(key/value)= **共通TVT軸グリッド**上の [typewell GR / 近傍井GR(v27と同じXY近傍) / 自分の既知帯GR]。
- クエリ = 評価ゾーン各行(GR窓は conv エンコーダが吸収, Z/md_since/prefix文脈)。
- **attention softmax = 各行の TVT 事後分布 → 事後平均が出力**(双峰なら2山に割れて自動hedge)。
- 損失 = 分布CE(真値TVT周りのソフトターゲット) + 事後平均MSE + 微調整TCN(final)のMSE。
- fold は v28/v29 と同一 GroupKFold(5)。PF由来の入力・特徴はゼロ。flip TTA 付き。
- cell2 の参照キャッシュ構築は初回のみ数分(refs_cache へ保存)。


In [ ]:
# ===== 設定 =====
import os, json, math, time, gc, copy, glob, warnings
import numpy as np, pandas as pd
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

PROJECT  = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
DATA_DIR = PROJECT / "rogii-wellbore-geology-prediction" / "train"
V28_ART  = PROJECT / "v28" / "artifacts_v28"
ART = PROJECT / "v31" / "artifacts_v31"; (ART / "nn").mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
# --- 参照グリッド / 近傍(v28 と同じ機構) ---
GRID_STEP = 0.5; GRID_PAD = 20.0
MAX_DIST = 1500.0; K_MAX = 5
PRE_LEN = 512
# --- モデル ---
D_MODEL = 128; DROP = 0.10; CTX_DIM = 64
SOFT_SIG = 2.0        # CE ソフトターゲットの幅(ft)
W_CE = 1.0; W_PM = 0.3; W_FIN = 0.7   # 損失重み(CE / 事後平均MSE / refiner後MSE)
REFINE = True
# --- 学習 ---
SEEDS = [0, 1]; EPOCHS = 40; LR = 3e-4; WD = 1e-4
BATCH_WELLS = 4; CROP = 1536          # クエリ側の学習時クロップ
NOISE = 0.02; EMA_DECAY = 0.999; CLIP = 1.0; WARM_FRAC = 0.05
FLIP = True                            # クエリ系列の反転aug+TTA(参照はそのまま)
EVAL_BS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)


In [ ]:
# ===== 参照キャッシュ構築(typewell/近傍/自分prefix を共通TVT軸へ, 初回のみ) =====
def _robust_norm(v):
    v = np.asarray(v, np.float64); f = np.isfinite(v)
    if f.sum() < 3: return np.zeros_like(v, dtype=np.float32)
    med = np.median(v[f]); iqr = max(np.percentile(v[f], 75) - np.percentile(v[f], 25), 1e-6)
    out = np.clip((v - med) / iqr, -8, 8)
    return out.astype(np.float32)

def _bin_curve(tvt, gr, gmin, G):
    idx = np.floor((np.asarray(tvt, np.float64) - gmin) / GRID_STEP).astype(int)
    gr = np.asarray(gr, np.float64)
    m = (idx >= 0) & (idx < G) & np.isfinite(gr) & np.isfinite(idx)
    s = np.zeros(G); c = np.zeros(G)
    np.add.at(s, idx[m], gr[m]); np.add.at(c, idx[m], 1)
    out = np.full(G, np.nan); nz = c > 0; out[nz] = s[nz] / c[nz]
    return out

cache_fp = ART / "refs_cache.npz"
wells = sorted({Path(p).stem.replace("__horizontal_well", "")
                for p in glob.glob(str(DATA_DIR / "*__horizontal_well.csv"))})
if cache_fp.exists():
    _z = np.load(cache_fp, allow_pickle=True)
    WD_ = {w: _z[w].item() for w in _z.files}
    print("refs cache load:", len(WD_), "wells")
else:
    t0 = time.time()
    raw = {}; cents = {}
    for w in wells:
        hw = pd.read_csv(DATA_DIR / f"{w}__horizontal_well.csv")
        tw = pd.read_csv(DATA_DIR / f"{w}__typewell.csv").sort_values("TVT")
        if "TVT" not in hw.columns or len(tw) < 3: continue
        kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
        if len(ev) == 0 or len(kn) < 10: continue
        raw[w] = (hw, tw, kn, ev)
        cents[w] = (float(hw["X"].mean()), float(hw["Y"].mean()))
    wid_all = list(raw); XY = np.array([cents[w] for w in wid_all])
    WD_ = {}
    for w in wid_all:
        hw, tw, kn, ev = raw[w]
        tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
        gmin = float(np.nanmin(tw_tvt)) - GRID_PAD
        G = int(np.ceil((float(np.nanmax(tw_tvt)) + GRID_PAD - gmin) / GRID_STEP)) + 1
        grid_tvt = gmin + GRID_STEP * np.arange(G)
        # typewell
        twc = _bin_curve(tw_tvt, tw_gr, gmin, G)
        # 近傍(XY重心, 自分除外, MAX_DIST, K_MAX) — 近傍の全行 (TVT, GR)
        d = np.hypot(XY[:, 0] - cents[w][0], XY[:, 1] - cents[w][1])
        order = np.argsort(d)
        nbr_curves = []
        for j in order:
            w2 = wid_all[j]
            if w2 == w or d[j] > MAX_DIST: continue
            hw2 = raw[w2][0]
            m2 = hw2["TVT"].notna() & hw2["GR"].notna()
            if m2.sum() < 30: continue
            nbr_curves.append(_bin_curve(hw2.loc[m2, "TVT"], hw2.loc[m2, "GR"], gmin, G))
            if len(nbr_curves) >= K_MAX: break
        # 自分の既知帯(prefix)
        kn2 = kn.dropna(subset=["MD", "Z", "GR", "TVT_input"])
        selfc = _bin_curve(kn2["TVT_input"], kn2["GR"], gmin, G) if len(kn2) >= 8 else np.full(G, np.nan)
        # クエリ(評価行)
        gr_full = hw["GR"].astype(float).interpolate(limit_direction="both")
        gr_full = gr_full.fillna(float(np.nanmean(tw_gr))).to_numpy(float)
        ev_pos = hw.index.get_indexer(ev.index)
        lk = float(kn["TVT_input"].iloc[-1]); md0 = float(kn["MD"].iloc[-1]); z0 = float(kn["Z"].iloc[-1])
        pre = kn2.tail(PRE_LEN)
        WD_[w] = dict(grid_tvt=grid_tvt.astype(np.float32), tw=twc, self=selfc,
                      nbr=np.stack(nbr_curves) if nbr_curves else np.zeros((0, G)),
                      q_gr=gr_full[ev_pos].astype(np.float32),
                      q_z=ev["Z"].to_numpy(np.float32), q_md=ev["MD"].to_numpy(np.float32),
                      lk=lk, md0=md0, z0=z0,
                      pre=pre[["MD", "Z", "GR", "TVT_input"]].to_numpy(np.float64) if len(pre) >= 8 else None,
                      tvt_true=ev["TVT"].to_numpy(np.float64))
    np.savez_compressed(cache_fp, **{w: np.array(v, dtype=object) for w, v in WD_.items()})
    print(f"refs built: {len(WD_)} wells ({time.time()-t0:.0f}s)")


In [ ]:
# ===== テンソル化(正規化, 参照ch/クエリch/prefix) + fold =====
REF_C = 7   # [tw_n, self_n, self_cov, nbr_mean_n, nbr_std, nbr_cov, tvt_rel]
Q_C   = 4   # [gr_n, dgr, z_rel, md_since/1000]
PRE_C = 4

WELLS = sorted(WD_)
DATA = {}
for w in WELLS:
    d = WD_[w]
    G = len(d["grid_tvt"]); T = len(d["q_gr"])
    tw_n = _robust_norm(d["tw"]); tw_n[~np.isfinite(d["tw"])] = 0.0
    self_cov = np.isfinite(d["self"]).astype(np.float32)
    self_n = _robust_norm(d["self"]); self_n[self_cov == 0] = 0.0
    if len(d["nbr"]):
        nb = d["nbr"]; cov = np.isfinite(nb)
        nb_n = np.stack([_robust_norm(r) for r in nb]); nb_n[~cov] = np.nan
        nbr_mean = np.nanmean(nb_n, 0); nbr_std = np.nanstd(nb_n, 0)
        nbr_cov = cov.mean(0).astype(np.float32)
        nbr_mean = np.nan_to_num(nbr_mean, nan=0.0); nbr_std = np.nan_to_num(nbr_std, nan=0.0)
    else:
        nbr_mean = np.zeros(G, np.float32); nbr_std = np.zeros(G, np.float32); nbr_cov = np.zeros(G, np.float32)
    tvt_rel = ((d["grid_tvt"] - d["lk"]) / 50.0).astype(np.float32)
    R = np.stack([tw_n, self_n, self_cov, nbr_mean.astype(np.float32),
                  nbr_std.astype(np.float32), nbr_cov, tvt_rel], 1)   # (G, REF_C)
    gr_n = _robust_norm(d["q_gr"])
    dgr = np.zeros(T, np.float32); dgr[1:] = np.diff(gr_n)
    z_rel = ((d["q_z"] - d["z0"]) / 50.0).astype(np.float32)
    mds = ((d["q_md"] - d["md0"]) / 1000.0).astype(np.float32)
    Q = np.stack([gr_n, dgr, z_rel, np.clip(mds, 0, None)], 1)        # (T, Q_C)
    if d["pre"] is not None:
        a = d["pre"]
        P_ = np.stack([_robust_norm(a[:, 2]), (a[:, 3] - a[-1, 3]) / 25.0,
                       (a[:, 1] - a[-1, 1]) / 50.0, (a[:, 0] - a[-1, 0]) / 1000.0], 1).astype(np.float32)
        P_ = np.nan_to_num(P_, nan=0.0, posinf=8.0, neginf=-8.0)
    else:
        P_ = np.zeros((1, PRE_C), np.float32)
    DATA[w] = dict(R=np.nan_to_num(R, nan=0.0).astype(np.float32), Q=np.nan_to_num(Q, nan=0.0),
                   P=P_, gt=d["grid_tvt"].astype(np.float32),
                   y=d["tvt_true"].astype(np.float64), lk=float(d["lk"]))
rows_total = sum(len(DATA[w]["y"]) for w in WELLS)
g_all = np.concatenate([[w] * len(DATA[w]["y"]) for w in WELLS])
folds_w = [(pd.unique(g_all[tr]), pd.unique(g_all[va]))
           for tr, va in GroupKFold(n_splits=N_SPLITS).split(np.zeros(len(g_all)), groups=g_all)]
print("wells", len(WELLS), "| rows", rows_total,
      "| nbrあり", sum(1 for w in WELLS if WD_[w]["nbr"].shape[0] > 0))


In [ ]:
# ===== モデル: RefEnc / QueryEnc / PrefixEnc / CrossAligner(+TCN refiner) =====
class ChanLN(nn.Module):
    def __init__(self, c):
        super().__init__(); self.ln = nn.LayerNorm(c)
    def forward(self, x):
        return self.ln(x.transpose(1, 2)).transpose(1, 2)

class Enc1D(nn.Module):
    """(B,C,L)->(B,D,L) 非因果conv 3層(局所窓の埋め込み)。"""
    def __init__(self, c_in, d, drop):
        super().__init__()
        self.c1 = nn.Conv1d(c_in, d, 5, padding=2); self.n1 = ChanLN(d)
        self.c2 = nn.Conv1d(d, d, 5, padding=8, dilation=4); self.n2 = ChanLN(d)
        self.c3 = nn.Conv1d(d, d, 5, padding=32, dilation=16); self.n3 = ChanLN(d)
        self.do = nn.Dropout(drop)
    def forward(self, x, m):
        h = self.n1(F.gelu(self.c1(x * m))) * m
        h = h + self.do(self.n2(F.gelu(self.c2(h)))) * m
        h = h + self.do(self.n3(F.gelu(self.c3(h)))) * m
        return h

class PrefixEnc(nn.Module):
    def __init__(self, c_in, ctx):
        super().__init__()
        self.c1 = nn.Conv1d(c_in, 64, 5, padding=2)
        self.c2 = nn.Conv1d(64, 64, 5, padding=8, dilation=4)
        self.fc = nn.Linear(128, ctx)
    def forward(self, x, m):
        h = F.gelu(self.c1(x * m)); h = F.gelu(self.c2(h)) * m
        s = m.sum(-1)
        mean = h.sum(-1) / s.clamp(min=1.0)
        mx = (h + (m - 1.0) * 1e4).amax(-1)
        mx = torch.where(s > 0, mx, torch.zeros_like(mx))
        return self.fc(torch.cat([mean, mx], dim=-1)) * (s > 0).float()

class RefineTCN(nn.Module):
    """事後統計(+クエリ埋め込み縮約)から補正deltaを出す軽いTCN。"""
    def __init__(self, c_in, d=64, drop=0.1):
        super().__init__()
        self.inp = nn.Conv1d(c_in, d, 1)
        self.b1 = nn.Conv1d(d, d, 5, padding=4, dilation=2)
        self.b2 = nn.Conv1d(d, d, 5, padding=16, dilation=8)
        self.b3 = nn.Conv1d(d, d, 5, padding=64, dilation=32)
        self.n1 = ChanLN(d); self.n2 = ChanLN(d); self.n3 = ChanLN(d)
        self.do = nn.Dropout(drop); self.out = nn.Conv1d(d, 1, 1)
    def forward(self, x, m):
        h = self.inp(x * m)
        h = (h + self.do(self.n1(F.gelu(self.b1(h))))) * m
        h = (h + self.do(self.n2(F.gelu(self.b2(h))))) * m
        h = (h + self.do(self.n3(F.gelu(self.b3(h))))) * m
        return self.out(h)[:, 0]   # (B,T)

class Aligner(nn.Module):
    def __init__(self, d=D_MODEL, drop=DROP, refine=REFINE):
        super().__init__()
        self.qenc = Enc1D(Q_C, d, drop); self.renc = Enc1D(REF_C, d, drop)
        self.pre = PrefixEnc(PRE_C, CTX_DIM); self.ctxp = nn.Linear(CTX_DIM, d)
        self.scale = d ** -0.5
        self.refine = RefineTCN(4 + 8) if refine else None
        self.qred = nn.Conv1d(d, 8, 1)   # refiner へ渡すクエリ縮約
    def forward(self, q, qm, r, rm, gt, px, pm):
        # q:(B,QC,T) qm:(B,1,T) r:(B,RC,G) rm:(B,1,G) gt:(B,G)=grid_tvt px/pm=prefix
        Qe = self.qenc(q, qm)                                # (B,D,T)
        ctx = self.ctxp(self.pre(px, pm))                    # (B,D)
        Qe = (Qe + ctx[:, :, None]) * qm
        Re = self.renc(r, rm)                                # (B,D,G)
        logits = torch.einsum("bdt,bdg->btg", Qe, Re) * self.scale
        logits = logits + (rm[:, 0, :][:, None, :] - 1.0) * 1e4
        A = torch.softmax(logits.float(), dim=-1)            # (B,T,G) = TVT事後分布
        pmn = torch.einsum("btg,bg->bt", A, gt.float())      # 事後平均
        m2 = torch.einsum("btg,bg->bt", A, (gt.float() ** 2))
        std = (m2 - pmn ** 2).clamp_min(1e-6).sqrt()
        ent = -(A * (A.clamp_min(1e-12)).log()).sum(-1)
        pk = A.amax(-1)
        if self.refine is not None:
            feats = torch.stack([pmn / 50.0 - pmn[:, :1] / 50.0, std / 50.0, ent, pk], 1)
            feats = torch.cat([feats.to(Qe.dtype), self.qred(Qe)], 1)
            delta = self.refine(feats, qm) * 10.0
            fin = pmn + delta.float()
        else:
            fin = pmn
        return A, pmn, fin, std, ent

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow: self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items(): sd[k].copy_(v)

def make_batch(wids, train, rng, flip_all=False):
    sel = []
    for w in wids:
        T = len(DATA[w]["y"]); sl = np.arange(T)
        if flip_all or (train and FLIP and rng.random() < 0.5): sl = sl[::-1]
        if train and T > CROP:
            s = int(rng.integers(0, T - CROP + 1)); sl = sl[s:s + CROP]
        sel.append((w, sl))
    B = len(sel)
    Tm = max(len(sl) for _, sl in sel); Gm = max(len(DATA[w]["gt"]) for w, _ in sel)
    Pm = max(len(DATA[w]["P"]) for w, _ in sel)
    q = np.zeros((B, Tm, Q_C), np.float32); qm = np.zeros((B, Tm), np.float32)
    r = np.zeros((B, Gm, REF_C), np.float32); rm = np.zeros((B, Gm), np.float32)
    gt = np.zeros((B, Gm), np.float32); yb = np.full((B, Tm), np.nan, np.float32)
    px = np.zeros((B, Pm, PRE_C), np.float32); pxm = np.zeros((B, Pm), np.float32)
    for i, (w, sl) in enumerate(sel):
        d = DATA[w]; L = len(sl); G = len(d["gt"])
        q[i, :L] = d["Q"][sl]; qm[i, :L] = 1.0
        r[i, :G] = d["R"]; rm[i, :G] = 1.0
        gt[i, :G] = d["gt"]; gt[i, G:] = d["gt"][-1]
        yb[i, :L] = d["y"][sl]
        px[i, :len(d["P"])] = d["P"]; pxm[i, :len(d["P"])] = 1.0
    to = lambda a: torch.from_numpy(a).to(DEVICE)
    return (to(q).transpose(1, 2), to(qm).unsqueeze(1), to(r).transpose(1, 2), to(rm).unsqueeze(1),
            to(gt), to(yb), to(px).transpose(1, 2), to(pxm).unsqueeze(1), sel)

def loss_fn(A, pmn, fin, yb, gt, qm):
    ok = torch.isfinite(yb) & (qm[:, 0, :] > 0.5)
    y = torch.where(ok, yb, torch.zeros_like(yb))
    # ソフトターゲットCE: N(y, SOFT_SIG) をグリッド上で離散化
    tgt = torch.exp(-0.5 * ((gt[:, None, :] - y[:, :, None]) / SOFT_SIG) ** 2)
    tgt = tgt / tgt.sum(-1, keepdim=True).clamp_min(1e-12)
    ce = -(tgt * A.clamp_min(1e-12).log()).sum(-1)
    okf = ok.float()
    l_ce = (ce * okf).sum() / okf.sum().clamp_min(1.0)
    l_pm = (((pmn - y) / 25.0) ** 2 * okf).sum() / okf.sum().clamp_min(1.0)
    l_fin = (((fin - y) / 25.0) ** 2 * okf).sum() / okf.sum().clamp_min(1.0)
    return W_CE * l_ce + W_PM * l_pm + W_FIN * l_fin

@torch.no_grad()
def predict_wells(model, wids):
    model.eval(); outs = {}
    def run(flip):
        res = {}
        for i in range(0, len(wids), EVAL_BS):
            sub = wids[i:i + EVAL_BS]
            q, qm, r, rm, gt, yb, px, pxm, sel = make_batch(sub, False, None, flip_all=flip)
            with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
                A, pmn, fin, std, ent = model(q, qm, r, rm, gt, px, pxm)
            fin = fin.float().cpu().numpy()
            for b, (w, sl) in enumerate(sel):
                p = fin[b, :len(sl)]
                res[w] = p[::-1].copy() if flip else p
        return res
    o1 = run(False)
    if FLIP:
        o2 = run(True)
        for w in wids: outs[w] = 0.5 * (o1[w] + o2[w])
    else:
        outs = o1
    return outs


In [ ]:
# ===== 学習: SEEDS × 5fold(既存 .pt skip) =====
def cosine_lr(step, total, warm):
    if step < warm: return step / max(1, warm)
    t = (step - warm) / max(1, total - warm)
    return 0.5 * (1.0 + math.cos(math.pi * t))

def train_fold(seed, k):
    trw, vaw = folds_w[k]; trw = list(trw); vaw = list(vaw)
    fp = ART / "nn" / f"al_s{seed}_fold{k}.pt"
    model = Aligner().to(DEVICE)
    if fp.exists():
        model.load_state_dict(torch.load(fp, map_location=DEVICE))
        return predict_wells(model, vaw), "load"
    torch.manual_seed(1000 * seed + k); rng = np.random.default_rng(1000 * seed + k)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    steps_ep = math.ceil(len(trw) / BATCH_WELLS)
    total = steps_ep * EPOCHS; warm = int(total * WARM_FRAC)
    ema = EMA(model, EMA_DECAY); best = (1e18, None); step = 0
    for ep in range(EPOCHS):
        model.train(); order = rng.permutation(len(trw))
        for i in range(0, len(order), BATCH_WELLS):
            q, qm, r, rm, gt, yb, px, pxm, _ = make_batch([trw[j] for j in order[i:i + BATCH_WELLS]], True, rng)
            if NOISE: q = q + NOISE * torch.randn_like(q)
            for pg in opt.param_groups: pg["lr"] = LR * cosine_lr(step, total, warm)
            with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
                A, pmn, fin, std, ent = model(q, qm, r, rm, gt, px, pxm)
            loss = loss_fn(A, pmn.float(), fin.float(), yb, gt, qm)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step(); ema.update(model); step += 1
        bak = copy.deepcopy(model.state_dict()); ema.copy_to(model)
        pr = predict_wells(model, vaw)
        e = np.concatenate([pr[w] - DATA[w]["y"] for w in vaw]); e = e[np.isfinite(e)]
        rm_ = float(np.sqrt(np.mean(e ** 2)))
        if not np.isfinite(rm_):
            raise RuntimeError(f"val RMSE 非有限 (ep={ep})")
        if rm_ < best[0]: best = (rm_, copy.deepcopy(model.state_dict()))
        model.load_state_dict(bak)
        if (ep + 1) % 10 == 0: print(f"    ep{ep+1} val={rm_:.3f} (best {best[0]:.3f})", flush=True)
    model.load_state_dict(best[1]); torch.save(best[1], fp)
    return predict_wells(model, vaw), f"fit best={best[0]:.4f}"

PRED = {}
for seed in SEEDS:
    t0 = time.time()
    for k in range(N_SPLITS):
        pr, tag = train_fold(seed, k)
        for w, p in pr.items(): PRED.setdefault(w, []).append(p)
        va = folds_w[k][1]
        e = np.concatenate([pr[w] - DATA[w]["y"] for w in va]); e = e[np.isfinite(e)]
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
        print(f"seed{seed} fold{k} [{tag}] rmse={np.sqrt(np.mean(e**2)):.4f}", flush=True)
    print(f"== seed{seed} done ({time.time()-t0:.0f}s)")


In [ ]:
# ===== 評価: overall / wells>10 / 判定(合格ライン10) =====
ens = {w: np.mean(ps, 0) for w, ps in PRED.items()}
e_all = np.concatenate([ens[w] - DATA[w]["y"] for w in WELLS])
e_all = e_all[np.isfinite(e_all)]
overall = float(np.sqrt(np.mean(e_all ** 2)))
per = {w: float(np.sqrt(np.nanmean((ens[w] - DATA[w]["y"]) ** 2))) for w in WELLS}
n10 = sum(1 for v in per.values() if v > 10)
print(f"=== v31 aligner OOF overall = {overall:.4f}  wells>10 = {n10}/{len(WELLS)} ===")
print(f"判定: 床(PFフリーtabular)≈13 / **合格ライン < 10** / 参考: last_known複製=15.9, PF系フォーク群≈7.2")
np.savez_compressed(ART / "aligner_oof.npz", **{f"p_{w}": ens[w] for w in WELLS})
# 内訳: nbr あり/なし
has_nbr = {w: WD_[w]["nbr"].shape[0] > 0 for w in WELLS}
for flag, name in [(True, "nbrあり"), (False, "nbrなし(tw+selfのみ)")]:
    ws_ = [w for w in WELLS if has_nbr[w] == flag]
    if ws_:
        e = np.concatenate([ens[w] - DATA[w]["y"] for w in ws_]); e = e[np.isfinite(e)]
        print(f"  {name}: {len(ws_)}井戸 rmse={np.sqrt(np.mean(e**2)):.4f}")
